In [1]:
# Importação das bibliotecas
import pandas as pd
from sklearn import preprocessing as pp
from sklearn import linear_model as lm
from sklearn import metrics as mt
import numpy as np

In [2]:
def calculate_metrics(y, yhat):
    '''
        Recebe as targets original e prevista, calcula as métricas e retorna um dicionário
    '''
    mse = round( mt.mean_squared_error( y, yhat ), 3)
    metrics = {'R2': round( mt.r2_score( y, yhat ), 3),
           'MSE': mse,
           'RMSE': round( np.sqrt( mse ), 3),
           'MAE': round( mt.mean_absolute_error( y, yhat ), 3),
           'MAPE': round( mt.mean_absolute_percentage_error( y, yhat ), 3)}   

    return metrics

In [3]:
def best_values(df_scores):
    '''
        Recebe um dataframe com as métricas encontradas para cada parâmetro
        Retorna um dataframe o melhor valor de cada métrica, e qual parâmetro utilizado para encontrá-lo
    '''
    df_best_values = pd.DataFrame(columns=['metric', 'performance', 'degree', 'alpha', 'max_iter'])

    list_metrics = ['R2', 'MSE', 'RMSE', 'MAE', 'MAPE']
    for metric in list_metrics:
        if metric == 'R2':
            max_index = df_scores[metric].idxmax()
        else:
            max_index = df_scores[metric].idxmin()
        best_value = df_scores.loc[max_index, metric]
        best_degree = df_scores.loc[max_index, 'degree']
        best_alpha = df_scores.loc[max_index, 'alpha']
        best_max_iter = df_scores.loc[max_index, 'max_iter']
        df_best_values = df_best_values._append({'metric': metric, 'performance': best_value, 'degree': best_degree, 'alpha': best_alpha,
                                                'max_iter': best_max_iter}, ignore_index=True)

    return df_best_values
    

In [4]:
# Features selecionadas
features = ['song_duration_ms', 'acousticness', 'danceability', 'energy',
       'instrumentalness', 'key', 'liveness', 'loudness', 'audio_mode',
       'speechiness', 'tempo', 'time_signature', 'audio_valence']

In [5]:
# Carregamento dos dados de treino
X_train = pd.read_csv( 'X_training.csv' )
y_train = pd.read_csv( 'y_training.csv' )

# Preparação dos dados de treino
X_train = X_train.loc[:, features]
y_train = y_train.values.ravel()

In [6]:
# Carregamento dos dados de validação:
X_val = pd.read_csv( 'X_validation.csv' )
y_val = pd.read_csv( 'y_val.csv' )

# Preparação dos dados de validação
X_val = X_val.loc[:, features]
y_val = y_val.values.ravel()

In [7]:
# Carregamento dos dados de teste
X_test = pd.read_csv( 'X_test.csv' )
y_test = pd.read_csv( 'y_test.csv' )

# Preparação dos dados de teste
X_test = X_test.loc[:, features]
y_test = y_test.values.ravel()

In [8]:
# Treinamento do modelo com os parâmetros padrão do algoritmo

# Transformação polinomial do dataset
poly_features = pp.PolynomialFeatures()
X_train_poly = poly_features.fit_transform(X_train)
    
# Treinamento do modelo com os parâmetros padrão do algoritmo
model = lm.Lasso()
model.fit( X_train_poly, y_train )

,"alpha alpha: float, default=1.0Constant that multiplies the L1 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Lasso` object is not advised.Instead, you should use the :class:`LinearRegression` object.",1.0
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"precompute precompute: bool or array-like of shape (n_features, n_features), default=FalseWhether to use a precomputed Gram matrix to speed upcalculations. The Gram matrix can also be passed as argument.For sparse input this option is always ``False`` to preserve sparsity.",False
,"copy_X copy_X: bool, default=TrueIf ``True``, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=1000The maximum number of iterations.",1000
,"tol tol: float, default=1e-4The tolerance for the optimization: if the updates are smaller or equal to``tol``, the optimization code checks the dual gap for optimality and continuesuntil it is smaller or equal to ``tol``, see Notes below.",0.0001
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fit asinitialization, otherwise, just erase the previous solution.See :term:`the Glossary <warm_start>`.",False
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.",False
,"random_state random_state: int, RandomState instance, default=NoneThe seed of the pseudo random number generator that selects a randomfeature to update. Used when ``selection`` == 'random'.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",None
,"selection selection: {'cyclic', 'random'}, default='cyclic'If set to 'random', a random coefficient is updated every iterationrather than looping over features sequentially by default. This(setting to 'random') often leads to significantly faster convergenceespecially when tol is higher than 1e-4.",'cyclic'
Name,Type,Value


In [9]:
# Cálculo das métricas para os dados de treinamento com os parametros padrão, a objetivo de comparação
df_scores_train_ini = pd.DataFrame(columns=['R2', 'MSE', 'RMSE', 'MAE', 'MAPE'])

X_train_poly = poly_features.transform(X_train)
yhat_train = model.predict( X_train_poly)
res = calculate_metrics(y_train, yhat_train)
df_scores_train_ini = df_scores_train_ini._append({'R2': res['R2'], 'MSE': res['MSE'], 'RMSE': res['RMSE'],
                                               'MAE': res['MAE'], 'MAPE': res['MAPE']}, ignore_index=True)

df_scores_train_ini

,R2,MSE,RMSE,MAE,MAPE
0,0.009,473.639,21.763,17.285,8.7


In [10]:
# Cálculo das métricas para os dados de validação com os parametros padrão, a objetivo de comparação
df_scores_train_val = pd.DataFrame(columns=['R2', 'MSE', 'RMSE', 'MAE', 'MAPE'])

X_val_poly = poly_features.transform(X_val)
yhat_val = model.predict( X_val_poly)
res = calculate_metrics(y_val, yhat_val)
df_scores_train_val = df_scores_train_val._append({'R2': res['R2'], 'MSE': res['MSE'], 'RMSE': res['RMSE'],
                                               'MAE': res['MAE'], 'MAPE': res['MAPE']}, ignore_index=True)

df_scores_train_val

,R2,MSE,RMSE,MAE,MAPE
0,0.01,472.913,21.747,17.238,8.682


In [11]:
# Treinar os modelos variandos os parâmetros e armazenar os valores em dataframes

df_scores_params_train = pd.DataFrame(columns=['degree', 'R2', 'MSE', 'RMSE', 'MAE', 'MAPE'])
df_scores_params_val = pd.DataFrame(columns=['degree', 'R2', 'MSE', 'RMSE', 'MAE', 'MAPE'])

list_degrees = [1,2,3,4]
list_alpha = [1,5,10,25]
list_max_iter = [100,500,1000,5000]
for i in list_degrees:
    for j in list_alpha:
        for k in list_max_iter :

            # Transformação polinomial do dataset
            poly_features = pp.PolynomialFeatures(degree=i)
            X_train_poly = poly_features.fit_transform(X_train)
            
            # model training
            model = lm.Lasso( alpha=j, max_iter=k)
            model.fit( X_train_poly, y_train )
        
            # Coletando as métricas para os dados de treinamento
            X_train_poly = poly_features.transform(X_train)
            yhat_train = model.predict( X_train_poly)
            res = calculate_metrics(y_train, yhat_train)   
        
            df_scores_params_train = df_scores_params_train._append({'degree': i, 'alpha': j, 'max_iter': k, 'R2': res['R2'], 'MSE': res['MSE'],
                                                                     'RMSE': res['RMSE'], 'MAE': res['MAE'], 'MAPE': res['MAPE']}, ignore_index=True)
        
          
            # Coletando as métricas para os dados de validação
            X_val_poly = poly_features.transform(X_val) 
            yhat_val = model.predict( X_val_poly)
            res = calculate_metrics(y_val, yhat_val)
        
            df_scores_params_val = df_scores_params_val._append({'degree': i, 'alpha': j, 'max_iter': k, 'R2': res['R2'], 'MSE': res['MSE'],
                                                                     'RMSE': res['RMSE'], 'MAE': res['MAE'], 'MAPE': res['MAPE']}, ignore_index=True) 

/home/gabriel/miniconda3/envs/estudoml/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.221167e+06, tolerance: 5.042e+02
  model = cd_fast.enet_coordinate_descent(
/home/gabriel/miniconda3/envs/estudoml/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.607340e+05, tolerance: 5.042e+02
  model = cd_fast.enet_coordinate_descent(
/home/gabriel/miniconda3/envs/estudoml/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the

In [12]:
# Determinar as melhores performances para os dados de treino e validação
df_best_values_train = best_values(df_scores_params_train)
df_best_values_val = best_values(df_scores_params_val)
df_best_values_val

,metric,performance,degree,alpha,max_iter
0,R2,0.014,3.0,1.0,100.0
1,MSE,470.756,3.0,1.0,100.0
2,RMSE,21.697,3.0,1.0,100.0
3,MAE,17.181,3.0,1.0,100.0
4,MAPE,8.656,3.0,1.0,100.0


In [13]:
# Concatenar os dados de treino e validação para uso da técnica de validação holdout com os dados de teste
X_trainval = pd.concat([X_train, X_val], ignore_index=True)
y_trainval = np.concatenate((y_train, y_val), axis=None)

In [14]:
# Agora vamos medir a performance de teste individualmente para cada uma das métricas


# R2
best_degree = int(df_best_values_val[df_best_values_val['metric']=='R2']['degree'].item())
best_alpha = df_best_values_val[df_best_values_val['metric']=='R2']['alpha'].item()
best_max_iter = int(df_best_values_val[df_best_values_val['metric']=='R2']['max_iter'].item())

# Transformação polinomial do dataset
poly_features = pp.PolynomialFeatures(degree=best_degree)
X_trainval_poly = poly_features.fit_transform(X_trainval)

# Performance de teste:
model = lm.Lasso(alpha=best_alpha, max_iter=best_max_iter)
model.fit( X_trainval_poly, y_trainval )

X_test_poly = poly_features.transform(X_test) 
yhat_test = model.predict( X_test_poly)
res = calculate_metrics(y_test, yhat_test)
R2_test = res['R2']

In [15]:
# MSE
best_max_depth = int(df_best_values_val[df_best_values_val['metric']=='MSE']['degree'].item())
best_alpha = df_best_values_val[df_best_values_val['metric']=='R2']['alpha'].item()
best_max_iter = int(df_best_values_val[df_best_values_val['metric']=='R2']['max_iter'].item())

# Transformação polinomial do dataset
poly_features = pp.PolynomialFeatures(degree=best_degree)
X_trainval_poly = poly_features.fit_transform(X_trainval)

# Performance de teste:
model = lm.Lasso(alpha=best_alpha, max_iter=best_max_iter)
model.fit( X_trainval_poly, y_trainval )

X_test_poly = poly_features.transform(X_test) 
yhat_test = model.predict( X_test_poly)
res = calculate_metrics(y_test, yhat_test)
MSE_test = res['MSE']

In [16]:
# RMSE
best_max_depth = int(df_best_values_val[df_best_values_val['metric']=='RMSE']['degree'].item())
best_alpha = df_best_values_val[df_best_values_val['metric']=='R2']['alpha'].item()
best_max_iter = int(df_best_values_val[df_best_values_val['metric']=='R2']['max_iter'].item())

# Transformação polinomial do dataset
poly_features = pp.PolynomialFeatures(degree=best_degree)
X_trainval_poly = poly_features.fit_transform(X_trainval)

# Performance de teste:
model = lm.Lasso(alpha=best_alpha, max_iter=best_max_iter)
model.fit( X_trainval_poly, y_trainval )

X_test_poly = poly_features.transform(X_test) 
yhat_test = model.predict( X_test_poly)
res = calculate_metrics(y_test, yhat_test)
RMSE_test = res['RMSE']

In [17]:
# MAE
best_max_depth = int(df_best_values_val[df_best_values_val['metric']=='MAE']['degree'].item())
best_alpha = df_best_values_val[df_best_values_val['metric']=='R2']['alpha'].item()
best_max_iter = int(df_best_values_val[df_best_values_val['metric']=='R2']['max_iter'].item())

# Transformação polinomial do dataset
poly_features = pp.PolynomialFeatures(degree=best_degree)
X_trainval_poly = poly_features.fit_transform(X_trainval)

# Performance de teste:
model = lm.Lasso(alpha=best_alpha, max_iter=best_max_iter)
model.fit( X_trainval_poly, y_trainval )

X_test_poly = poly_features.transform(X_test) 
yhat_test = model.predict( X_test_poly)
res = calculate_metrics(y_test, yhat_test)
MAE_test = res['MAE']

In [18]:
# MAPE
best_max_depth = int(df_best_values_val[df_best_values_val['metric']=='MAPE']['degree'].item())
best_alpha = df_best_values_val[df_best_values_val['metric']=='R2']['alpha'].item()
best_max_iter = int(df_best_values_val[df_best_values_val['metric']=='R2']['max_iter'].item())

# Transformação polinomial do dataset
poly_features = pp.PolynomialFeatures(degree=best_degree)
X_trainval_poly = poly_features.fit_transform(X_trainval)

# Performance de teste:
model = lm.Lasso(alpha=best_alpha, max_iter=best_max_iter)
model.fit( X_trainval_poly, y_trainval )

X_test_poly = poly_features.transform(X_test) 
yhat_test = model.predict( X_test_poly)
res = calculate_metrics(y_test, yhat_test)
MAPE_test = res['MAPE']

In [19]:
# Construindo um dataframe com os valores finais de performance para facilitar a visualização

R2_train = df_best_values_train[df_best_values_train['metric'] == 'R2']['performance'].item()
MSE_train = df_best_values_train[df_best_values_train['metric'] == 'MSE']['performance'].item()
RMSE_train = df_best_values_train[df_best_values_train['metric'] == 'RMSE']['performance'].item()
MAE_train = df_best_values_train[df_best_values_train['metric'] == 'MAE']['performance'].item()
MAPE_train = df_best_values_train[df_best_values_train['metric'] == 'MAPE']['performance'].item()

R2_val = df_best_values_val[df_best_values_val['metric'] == 'R2']['performance'].item()
MSE_val = df_best_values_val[df_best_values_val['metric'] == 'MSE']['performance'].item()
RMSE_val = df_best_values_val[df_best_values_val['metric'] == 'RMSE']['performance'].item()
MAE_val = df_best_values_val[df_best_values_val['metric'] == 'MAE']['performance'].item()
MAPE_val = df_best_values_val[df_best_values_val['metric'] == 'MAPE']['performance'].item()


df_scores_final = pd.DataFrame(columns=['dataset','R2', 'MSE', 'RMSE', 'MAE', 'MAPE'])

df_scores_final = df_scores_final._append({'dataset': 'Treinamento', 'R2': R2_train, 'MSE': MSE_train,
                                          'RMSE': RMSE_train, 'MAE': MAE_train, 'MAPE': MAPE_train }, ignore_index=True) 

df_scores_final = df_scores_final._append({'dataset': 'Validação', 'R2': R2_val, 'MSE': MSE_val,
                                          'RMSE': RMSE_val, 'MAE': MAE_val, 'MAPE': MAPE_val}, ignore_index=True) 

df_scores_final = df_scores_final._append({'dataset': 'Teste', 'R2': R2_test, 'MSE': MSE_test,
                                          'RMSE': RMSE_test, 'MAE': MAE_test, 'MAPE': MAPE_test}, ignore_index=True) 


df_scores_final

,dataset,R2,MSE,RMSE,MAE,MAPE
0,Treinamento,0.016,470.496,21.691,17.210,8.590
1,Validação,0.014,470.756,21.697,17.181,8.656
2,Teste,-0.001,487.522,22.080,17.441,8.756
